In [7]:
import pandas as pd
import spacy
from spacy.pipeline import EntityRuler
from transformers import pipeline

In [8]:

# 1. LOAD PRETRAINED TRANSFORMER MODEL (Hugging Face)

try:
    # dslim/bert-base-NER detects standard entities: MISC, ORG, LOC, PER
    bert_ner = pipeline(
        "ner", model="dslim/bert-base-NER", aggregation_strategy="simple"
    )
    transformer_available = True
    print("✅ Pretrained BERT Transformer model loaded successfully!")
except Exception as e:
    print(f"⚠️ Could not load pretrained transformer model: {e}")
    print("Operating in offline fallback mode for Transformer pipeline.")
    transformer_available = False

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2387.06it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/dslim/bert-base-NER/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/dslim/bert-base-NER/resolve/main/processor_config.json
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/dslim/bert-base-NER/resolve/main/processor_config.json
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown wh

✅ Pretrained BERT Transformer model loaded successfully!


In [9]:
# ==============================================================================
# 2. SETUP YOUR CUSTOM SPACY PIPELINE WITH ENTITYRULER
# ==============================================================================
nlp_custom = spacy.load("en_core_web_sm")
ruler = nlp_custom.add_pipe("entity_ruler", before="ner")

# Complete patterns matching your custom dataset ontology
patterns = [
    # SKILL
    {"label": "SKILL", "pattern": "Python"},
    {"label": "SKILL", "pattern": "SQL"},
    {"label": "SKILL", "pattern": "Java"},
    {"label": "SKILL", "pattern": "Scala"},
    {
        "label": "SKILL",
        "pattern": [{"LOWER": "machine"}, {"LOWER": "learning"}],
    },
    {"label": "SKILL", "pattern": [{"LOWER": "deep"}, {"LOWER": "learning"}]},
    {
        "label": "SKILL",
        "pattern": [
            {"LOWER": "natural"},
            {"LOWER": "language"},
            {"LOWER": "processing"},
        ],
    },
    # CLOUD_PLATFORM
    {"label": "CLOUD_PLATFORM", "pattern": "AWS"},
    {"label": "CLOUD_PLATFORM", "pattern": "Azure"},
    {
        "label": "CLOUD_PLATFORM",
        "pattern": [{"LOWER": "google"}, {"LOWER": "cloud"}],
    },
    {
        "label": "CLOUD_PLATFORM",
        "pattern": [{"LOWER": "microsoft"}, {"LOWER": "azure"}],
    },
    # DATABASE
    {"label": "DATABASE", "pattern": "PostgreSQL"},
    {"label": "DATABASE", "pattern": "MongoDB"},
    {"label": "DATABASE", "pattern": "MySQL"},
    # TOOL
    {"label": "TOOL", "pattern": [{"LOWER": "power"}, {"LOWER": "bi"}]},
    {"label": "TOOL", "pattern": "Tableau"},
    {"label": "TOOL", "pattern": "Docker"},
    {"label": "TOOL", "pattern": "Excel"},
    {"label": "TOOL", "pattern": "Git"},
    # TECHNOLOGY
    {"label": "TECHNOLOGY", "pattern": "TensorFlow"},
    {"label": "TECHNOLOGY", "pattern": "Kubernetes"},
    {"label": "TECHNOLOGY", "pattern": "Spark"},
    {"label": "TECHNOLOGY", "pattern": "Hadoop"},
    {"label": "TECHNOLOGY", "pattern": "PyTorch"},
]
ruler.add_patterns(patterns)



In [10]:
# ==============================================================================
# 3. LOAD YOUR DATASET & EXTRACT SAMPLE DESCRIPTIONS
# ==============================================================================
# Update CSV path if running on a different environment
dataset_path = "C:/Users/HP/Desktop/Internship Final Submission/NLP/2_raw_jobs.csv"

try:
    df = pd.read_csv(dataset_path).dropna(subset=["job_description"])
    # Pick 5 sample descriptions from your CSV dataset
    test_sentences = df["job_description"].head(5).tolist()
    print(
        f"✅ Loaded {len(test_sentences)} sample job descriptions from dataset.\n"
    )
except Exception:
    print(
        "⚠️ Dataset file not found at path. Using dataset sample sentences.\n"
    )
    test_sentences = [
        "We need a Data Scientist with Python, AWS, PostgreSQL, and Power BI experience.",
        "Experience with Microsoft Azure, Google Cloud, and Docker is required.",
        "Candidate should know TensorFlow, Kubernetes, React, and MongoDB.",
        "Must have strong SQL skills and experience with Tableau and PyTorch.",
    ]



✅ Loaded 5 sample job descriptions from dataset.



In [11]:
# ==============================================================================
# 4. RUN COMPARISON: CUSTOM SPACY ENTITYRULER VS BERT TRANSFORMER
# ==============================================================================
print(
    "=========================================================================="
)
print("=== Custom spaCy EntityRuler vs Pretrained BERT-NER Transformer ===")
print(
    "==========================================================================\n"
)

custom_labels = {
    "SKILL",
    "TOOL",
    "TECHNOLOGY",
    "DATABASE",
    "CLOUD_PLATFORM",
}

for i, sentence in enumerate(test_sentences, 1):
    # Truncate output text if sentence is overly long
    display_text = (
        sentence[:120] + "..." if len(sentence) > 120 else sentence
    )
    print(f"Sample #{i}: {display_text}")

    # 1. Custom spaCy Extraction
    doc = nlp_custom(sentence)
    custom_result = [
        (e.text, e.label_) for e in doc.ents if e.label_ in custom_labels
    ]
    print(f"  🔹 Custom spaCy EntityRuler : {custom_result}")

    # 2. Transformer Extraction
    if transformer_available:
        # Truncate text to max 512 tokens for BERT model limits
        bert_output = bert_ner(sentence[:512])
        bert_result = [
            (r["word"], r["entity_group"], f"{r['score']:.2f}")
            for r in bert_output
        ]
        print(f"  🔸 Pretrained BERT-NER      : {bert_result}")

    print("-" * 75)

=== Custom spaCy EntityRuler vs Pretrained BERT-NER Transformer ===

Sample #1: We are looking for a motivated Supply Chain Coordinator to join our growing team. Responsibilities include: communicate ...
  🔹 Custom spaCy EntityRuler : []
  🔸 Pretrained BERT-NER      : []
---------------------------------------------------------------------------
Sample #2: Join us as a Backend Software Engineer and help drive our team's success. Responsibilities include: develop and maintain...
  🔹 Custom spaCy EntityRuler : [('Kubernetes', 'TECHNOLOGY'), ('Git', 'TOOL'), ('AWS', 'CLOUD_PLATFORM'), ('Java', 'SKILL')]
  🔸 Pretrained BERT-NER      : [('Ku', 'ORG', '0.30'), ('C', 'MISC', '0.87'), ('G', 'MISC', '0.63'), ('AWS', 'ORG', '0.63'), ('Java', 'MISC', '1.00'), ('Spring Boot', 'MISC', '0.90')]
---------------------------------------------------------------------------
Sample #3: We are looking for a motivated UX/UI Designer to join our growing team. Responsibilities include: collaborate with produ.

In [5]:
!pip install transformers torch

In [10]:
import torch
import transformers

print(f"Torch Version: {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")

Torch Version: 2.14.0+cpu
Transformers Version: 5.16.1
